# Bikeztagram — MiniMax Music 3 £0 Kaggle T4x2 Batch Forge Test

This is the unattended proof-of-life run. Use **Settings → Accelerator → T4 x2** and **Save & Run All**. The batch run downloads MiniMax Music 3, starts the private SGLang engine, renders a real WAV, saves it to `/kaggle/working`, then shuts the engine down. No Hugging Face hosted inference or ZeroGPU is used.

**Important:** Save & Run All creates a background run that can continue after the browser/app is closed. Keep Internet enabled.


In [ ]:
import os, subprocess, shutil, urllib.request
print('=== BIKEZTAGRAM GPU PREFLIGHT ===')
gpus=subprocess.check_output(['nvidia-smi','--query-gpu=index,name,memory.total','--format=csv,noheader'],text=True).strip()
print(gpus)
gpu_count=len([x for x in gpus.splitlines() if x.strip()])
if gpu_count < 2: raise RuntimeError(f'Need NVIDIA T4 x2; detected {gpu_count} GPU(s). Set Accelerator to T4 x2.')
os.makedirs('/kaggle/temp',exist_ok=True)
for p in ['/kaggle/working','/kaggle/temp']:
 u=shutil.disk_usage(p); print(p,'free GB:',round(u.free/1024**3,2))
urllib.request.urlopen('https://huggingface.co',timeout=10)
print('Internet: OK')


In [ ]:
import subprocess,sys,os
print('=== INSTALL PYTHON 3.12 + SGLANG-OMNI ===')
subprocess.run([sys.executable,'-m','pip','install','-q','--no-input','uv'],check=True)
subprocess.run([sys.executable,'-m','uv','python','install','3.12'],check=True)
venv='/kaggle/temp/music3-venv'
subprocess.run([sys.executable,'-m','uv','venv',venv,'--python','3.12'],check=True)
py=os.path.join(venv,'bin','python'); sgl=os.path.join(venv,'bin','sgl-omni')
subprocess.run([sys.executable,'-m','uv','pip','install','--python',py,'--prerelease=allow','sglang-omni==0.1.7'],check=True)
if not os.path.exists(sgl): raise RuntimeError('sgl-omni installation failed')
print('SGLang-Omni:',sgl); print('Engine Python:',subprocess.check_output([py,'--version'],text=True).strip())


In [ ]:
import os,subprocess,time,requests,signal,shutil
from pathlib import Path
print('=== UNATTENDED MINI MAX MUSIC 3 RENDER ===')
os.environ['HF_HOME']='/kaggle/temp/huggingface'
os.environ['HF_HUB_CACHE']='/kaggle/temp/huggingface/hub'
os.makedirs(os.environ['HF_HUB_CACHE'],exist_ok=True)
log_path='/kaggle/temp/music3_batch.log'
log=open(log_path,'w',buffering=1)
sgl='/kaggle/temp/music3-venv/bin/sgl-omni'
env=os.environ.copy()
p=subprocess.Popen(['bash','-lc',f'CUDA_VISIBLE_DEVICES=0,1 {sgl} serve --model-path MiniMaxAI/MiniMax-Music3 --port 8000'],stdout=log,stderr=subprocess.STDOUT,start_new_session=True,env=env)
print('Server PID:',p.pid)
ready=False
try:
 for i in range(240):
  r=subprocess.run(['curl','-s','-m','5','http://127.0.0.1:8000/v1/models'],capture_output=True,text=True)
  if r.returncode==0 and r.stdout:
   ready=True; print('SERVER READY after',i*5,'seconds'); break
  if i%6==0:
   print('still starting —',i*5,'seconds')
   try:
    lines=Path(log_path).read_text(errors='replace').splitlines(); print('\n'.join(lines[-8:]))
   except Exception: pass
  if p.poll() is not None:
   raise RuntimeError('Music 3 server exited during startup. Last log:\n'+Path(log_path).read_text(errors='replace')[-12000:])
  time.sleep(5)
 if not ready: raise RuntimeError('Music 3 server did not become ready within 20 minutes. Last log:\n'+Path(log_path).read_text(errors='replace')[-12000:])
 lyrics='''[Verse]\nNeon cuts across the road\nEngine singing through the night\nCity fading in the mirrors\nChasing every line of light\n\n[Chorus]\nRide into the open dark\nChase the road and chase the light\nNo turning back, the engine calls\nWe own the road tonight'''
 instructions='Cinematic electronic rock motorcycle soundtrack, 105 BPM, D minor, deep punchy drums, pulsing bass, distorted electric guitar, atmospheric synths, dramatic build and a huge energetic chorus, polished modern production.'
 payload={'model':'MiniMaxAI/MiniMax-Music3','input':lyrics,'instructions':instructions,'seed':1001,'max_new_tokens':750}
 print('Rendering real 30-second Music 3 test...')
 resp=requests.post('http://127.0.0.1:8000/v1/audio/speech',json=payload,timeout=1800)
 resp.raise_for_status()
 out='/kaggle/working/bikeztagram_minimax_music3_test.wav'
 open(out,'wb').write(resp.content)
 print('DONE:',out,'bytes:',len(resp.content))
 print('WAV exists:',os.path.exists(out))
finally:
 print('Shutting down private Music 3 engine...')
 try:
  os.killpg(os.getpgid(p.pid),signal.SIGTERM)
  time.sleep(3)
 except Exception as e: print('TERM:',e)
 if p.poll() is None:
  try: os.killpg(os.getpgid(p.pid),signal.SIGKILL)
  except Exception: pass
 log.close()
 print('Engine shutdown complete. GPU is no longer being used by the renderer.')


In [ ]:
import os,subprocess,shutil
out='/kaggle/working/bikeztagram_minimax_music3_test.wav'
print('=== FINAL RESULT ===')
print('WAV:',out)
print('Exists:',os.path.exists(out))
if os.path.exists(out): print('Size MB:',round(os.path.getsize(out)/1024**2,2))
print('GPU state after shutdown:')
print(subprocess.check_output(['nvidia-smi','--query-gpu=index,utilization.gpu,memory.used','--format=csv,noheader'],text=True))


## Success condition

A successful run ends with `Exists: True` and a non-zero WAV size. The WAV is saved in `/kaggle/working`, which Kaggle preserves with the notebook version. This is the real local MiniMax Music 3 render proof; after this we can move the same lifecycle into Bikeztagram's private Music Engine.
